<a href="https://colab.research.google.com/github/revak18/science_fair_2026/blob/main/science_fair_2026_DRAFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 1. Setup & Configuration
import os, re, json, logging, zipfile, shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
import pulp
import requests
from google.colab import drive

# Mount Drive
drive.mount("/content/drive")

@dataclass(frozen=True)
class Config:
    # Paths
    base_dir: Path = Path("/content/drive/My Drive/Science_Fair/ISEF_2026_Project")
    data_dir: Path = base_dir / "data"
    out_dir: Path = base_dir / "results"

    # Data Sources
    kaggle_dataset: str = "robikscube/hourly-energy-consumption"
    water_bag_url: str = "https://www.hydroshare.org/hsapi/resource/2f54448714554f83b9655da108f0fd3f/bag"
    water_csv_name: str = "Water_IMPACT-Data_Consumption_2020.csv"

    # Grid Mappings (Region -> Balancing Authority)
    region_to_ba: Dict[str, str] = None

    # Simulation Parameters
    regions: List[str] = None
    ai_levels: List[float] = None
    horizon_h: int = 48
    rho_reserve: float = 0.15      # 15% safety margin above base load
    alpha_non_gas: float = 0.60    # 60% of peak is non-gas (nuclear/hydro)
    batch_share: float = 0.70      # 70% of AI load is flexible
    n_trials: int = 5              # Trials per scenario
    seed: int = 42

def get_config() -> Config:
    return Config(
        region_to_ba={
            "PJME": "PJM", "COMED": "PJM", "AEP": "PJM", "DAYTON": "PJM", "DOM": "PJM",
            "PJM": "PJM", "EKPC": "PJM", "DUQ": "PJM"
        },
        regions=["PJME", "COMED", "AEP", "DAYTON", "DOM"],
        ai_levels=[0.10, 0.20, 0.30]
    )

# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
cfg = get_config()
cfg.data_dir.mkdir(parents=True, exist_ok=True)
cfg.out_dir.mkdir(parents=True, exist_ok=True)
print(f"Project initialized at {cfg.base_dir}")

Mounted at /content/drive
Project initialized at /content/drive/My Drive/Science_Fair/ISEF_2026_Project
